In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week5-lessons2"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
orders_df = spark.read \
.format('csv') \
.option('header','true') \
.option('inferSchema','true') \
.load('/public/trendytech/orders_wh')

In [3]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



In [4]:
orders_df.show(3)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
+--------+--------------------+-----------+---------------+
only showing top 3 rows



## Using Dataframe APIS

In [5]:
## Top 15 customers who placed most number of orders

In [6]:
order_count = orders_df.groupBy("customer_id").count().sort("count",ascending = False).limit(15)

In [7]:
order_count.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       5897|   16|
|      12431|   16|
|        569|   16|
|       6316|   16|
|      12284|   15|
|       4320|   15|
|       5624|   15|
|       5283|   15|
|        221|   15|
|       5654|   15|
|       6248|   14|
|       3708|   14|
|       1011|   14|
|       8652|   14|
|       4517|   14|
+-----------+-----+



In [8]:
## Find the number of orders under each status

In [9]:
stat_count = orders_df.groupBy("order_status").count().sort("count",ascending = False).limit(15)

In [10]:
stat_count.show()

+---------------+-----+
|   order_status|count|
+---------------+-----+
|       COMPLETE|22899|
|PENDING_PAYMENT|15030|
|     PROCESSING| 8275|
|        PENDING| 7610|
|         CLOSED| 7556|
|        ON_HOLD| 3798|
|SUSPECTED_FRAUD| 1558|
|       CANCELED| 1428|
| PAYMENT_REVIEW|  729|
+---------------+-----+



In [11]:
## number of active customers

In [12]:
act_cust = orders_df.select("customer_id").distinct().count()

In [13]:
print(act_cust)

12405


In [14]:
## customer who has most number of closed orders

In [15]:
most_closed_cust = orders_df.filter("order_status = 'CLOSED' ").groupBy("customer_id").count().sort("count",ascending = False).limit(15)

In [16]:
most_closed_cust.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       1833|    6|
|       1363|    5|
|       1687|    5|
|       5493|    5|
|       2236|    4|
|      10018|    4|
|       2774|    4|
|       3631|    4|
|      12431|    4|
|       2403|    4|
|       4573|    4|
|       7948|    4|
|      10263|    4|
|        437|    4|
|       4588|    4|
+-----------+-----+



## Using SPARK SQL API

In [33]:
orders_df.createOrReplaceTempView("orders")

In [34]:
spark.sql("select count(*) from orders")

count(1)
68883


In [35]:
## Top 15 customers who placed most number of orders

In [36]:
spark.sql("select customer_id, count(*) as count from orders group by customer_id order by count desc").show(15)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|        569|   16|
|       5897|   16|
|      12431|   16|
|       6316|   16|
|        221|   15|
|       4320|   15|
|       5654|   15|
|      12284|   15|
|       5283|   15|
|       5624|   15|
|       3708|   14|
|       4517|   14|
|       6248|   14|
|       3710|   14|
|        791|   14|
+-----------+-----+
only showing top 15 rows



In [37]:
## Find the number of orders under each status

In [44]:
spark.sql("select order_status, count(distinct order_id) as count from orders group by order_status order by count desc").show()

+---------------+-----+
|   order_status|count|
+---------------+-----+
|       COMPLETE|22899|
|PENDING_PAYMENT|15030|
|     PROCESSING| 8275|
|        PENDING| 7610|
|         CLOSED| 7556|
|        ON_HOLD| 3798|
|SUSPECTED_FRAUD| 1558|
|       CANCELED| 1428|
| PAYMENT_REVIEW|  729|
+---------------+-----+



In [45]:
spark.sql("select order_status, count(*) as count from orders group by order_status order by count desc").show()

+---------------+-----+
|   order_status|count|
+---------------+-----+
|       COMPLETE|22899|
|PENDING_PAYMENT|15030|
|     PROCESSING| 8275|
|        PENDING| 7610|
|         CLOSED| 7556|
|        ON_HOLD| 3798|
|SUSPECTED_FRAUD| 1558|
|       CANCELED| 1428|
| PAYMENT_REVIEW|  729|
+---------------+-----+



In [40]:
## number of active customers

In [41]:
spark.sql("select count(distinct customer_id) from orders").show(15)

+---------------------------+
|count(DISTINCT customer_id)|
+---------------------------+
|                      12405|
+---------------------------+



In [42]:
## customer who has most number of closed orders

In [43]:
spark.sql("select customer_id, count(*) as count from orders where order_status = 'CLOSED' group by customer_id order by count desc").show(15)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       1833|    6|
|       1687|    5|
|       1363|    5|
|       5493|    5|
|      10018|    4|
|       2236|    4|
|       7948|    4|
|       7850|    4|
|       7879|    4|
|       4588|    4|
|       2403|    4|
|       3631|    4|
|      12431|    4|
|       2768|    4|
|       1521|    4|
+-----------+-----+
only showing top 15 rows

